- An Output Parser does 2 things:
  - Tell LLM how to format output
  - Parse the response into usable Python objects

# Types of Output Parsers
___
### 1.StrOutputParser (Basic)
     

- Just returns raw string.
```python
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

chain = llm | parser
print(chain.invoke("Explain Kafka in 2 lines"))


___
### 2. CommaSeparatedListOutputParser

- For simple lists.

```python
from langchain.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()

result = parser.parse("apple, banana, mango")
print(result)

___
### 3. StructuredOutputParser

- You define schema → LLM must follow it.

```python
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

schemas = [
    ResponseSchema(name="name", description="name of person"),
    ResponseSchema(name="age", description="age of person")
]

parser = StructuredOutputParser.from_response_schemas(schemas)

format_instructions = parser.get_format_instructions()


- Now inject into prompt:

```python
from langchain.prompts import PromptTemplate

prompt = PromptTemplate(
    template="Give person details\n{format_instructions}",
    input_variables=[],
    partial_variables={"format_instructions": format_instructions}
)

response = llm.invoke(prompt.format())
parsed = parser.parse(response)

print(parsed)

___

### 4. PydanticOutputParser (BEST PRACTICE )

- This is the most powerful and commonly used.
- You define a Pydantic model, and parser enforces it.

```python
from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser

class Person(BaseModel):
    name: str
    age: int

parser = PydanticOutputParser(pydantic_object=Person)

prompt = PromptTemplate(
    template="Give person details\n{format_instructions}",
    input_variables=[],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)
response = llm.invoke(prompt.format())
parsed = parser.parse(response)
print(parsed)

- Strong typing
- Validation
- Production ready

___

### 5.JSON Output Parser

- If you only need JSON:

```python
from langchain.output_parsers import JsonOutputParser

parser = JsonOutputParser()